# 01 | FastAPI Depends：把重复准备工作交给依赖注入

写 FastAPI 接口时，很快会遇到一类重复代码：分页参数每个列表接口都要解析，当前用户每个受保护接口都要检查，数据库会话每个读写接口都要准备和关闭。

`Depends` 解决的就是这件事：**让接口函数只声明“我需要什么”，具体怎么准备，由 FastAPI 在每次请求进来时自动完成。**

本文只回答一个问题：看到 `from fastapi import Depends` 之后，应该怎样读懂它、写出第一个可维护的依赖？

官方文档推荐在 Python 3.10+ 中使用 `Annotated[..., Depends(...)]` 的写法。下面的示例都采用这种形式。参考：

- FastAPI Dependencies: https://fastapi.tiangolo.com/tutorial/dependencies/
- Testing Dependencies with Overrides: https://fastapi.tiangolo.com/advanced/testing-dependencies/

如果当前环境还没有 FastAPI，可以先安装：

```bash
pip install "fastapi[standard]"
```

## 一、先看不用 Depends 会怎样

假设我们有两个列表接口，都需要 `skip` 和 `limit`。不用依赖注入时，参数和校验逻辑会散落在每个接口里。刚开始还好，接口一多就容易出现默认值不一致、校验规则不一致的问题。

In [ ]:
from fastapi import FastAPI, HTTPException, Query
from fastapi.testclient import TestClient

app_without_depends = FastAPI()


@app_without_depends.get("/articles")
def list_articles(skip: int = 0, limit: int = 10):
    if skip < 0:
        raise HTTPException(status_code=400, detail="skip 不能小于 0")
    if limit < 1 or limit > 50:
        raise HTTPException(status_code=400, detail="limit 必须在 1 到 50 之间")
    return {"resource": "articles", "skip": skip, "limit": limit}


@app_without_depends.get("/comments")
def list_comments(skip: int = 0, limit: int = 10):
    if skip < 0:
        raise HTTPException(status_code=400, detail="skip 不能小于 0")
    if limit < 1 or limit > 50:
        raise HTTPException(status_code=400, detail="limit 必须在 1 到 50 之间")
    return {"resource": "comments", "skip": skip, "limit": limit}

这段代码的问题不是“不能跑”，而是重复。重复代码最麻烦的地方，通常不是多写几行，而是未来某一次只改了其中一处。

## 二、Depends 的最小模型

一个依赖函数，看起来很像一个没有路由装饰器的接口函数：它可以声明查询参数、请求头、路径参数，也可以返回任意对象。

`Depends(get_pagination)` 的意思不是立刻调用 `get_pagination()`。它是在告诉 FastAPI：每次请求这个接口前，请先按规则调用 `get_pagination`，然后把返回值放进 `pagination` 这个参数里。

In [ ]:
from typing import Annotated

from fastapi import Depends, FastAPI, Query
from fastapi.testclient import TestClient

app = FastAPI()


def get_pagination(
    skip: Annotated[int, Query(ge=0)] = 0,
    limit: Annotated[int, Query(ge=1, le=50)] = 10,
) -> dict:
    return {"skip": skip, "limit": limit}


@app.get("/articles")
def list_articles(pagination: Annotated[dict, Depends(get_pagination)]):
    return {"resource": "articles", **pagination}


@app.get("/comments")
def list_comments(pagination: Annotated[dict, Depends(get_pagination)]):
    return {"resource": "comments", **pagination}


client = TestClient(app)

print(client.get("/articles?skip=20&limit=5").json())
print(client.get("/comments").json())
print(client.get("/articles?skip=-1").status_code)

这里有两个细节值得慢慢看：

- `get_pagination` 的参数仍然由 FastAPI 从请求里解析，所以 `Query(ge=0)` 这些校验依然有效。
- 路由函数拿到的是依赖函数的返回值，不需要知道它从哪里来。

换句话说，`Depends` 让“请求解析”和“业务处理”之间多了一层可复用的准备步骤。

## 三、用类型别名减少样板代码

当同一个依赖在很多接口里复用时，可以把 `Annotated[类型, Depends(...)]` 保存成一个类型别名。它仍然是普通 Python 写法，只是刚好很适合 FastAPI。

In [ ]:
Pagination = Annotated[dict, Depends(get_pagination)]


@app.get("/tags")
def list_tags(pagination: Pagination):
    return {"resource": "tags", **pagination}


client = TestClient(app)
client.get("/tags?limit=3").json()

类型别名的价值在大项目里更明显：接口函数会更短，编辑器也更容易保留类型信息。

## 四、依赖不只适合参数复用，也适合检查当前用户

`Depends` 很常见的用途是认证和权限。下面用一个极简 token 示例模拟“从请求头里解析当前用户”。真实项目里你会换成数据库查询、JWT 校验或 OAuth2 流程，但结构是一样的。

In [ ]:
from fastapi import Header, HTTPException
from pydantic import BaseModel


class CurrentUser(BaseModel):
    username: str
    role: str


def get_current_user(
    authorization: Annotated[str | None, Header()] = None,
) -> CurrentUser:
    if authorization != "Bearer dev-token":
        raise HTTPException(status_code=401, detail="无效的访问令牌")
    return CurrentUser(username="alice", role="admin")


UserDep = Annotated[CurrentUser, Depends(get_current_user)]


@app.get("/me")
def read_me(user: UserDep):
    return {"username": user.username, "role": user.role}


client = TestClient(app)

print(client.get("/me").status_code)
print(client.get("/me", headers={"Authorization": "Bearer dev-token"}).json())

注意这时候接口函数 `read_me` 没有写任何 header 解析逻辑。它只说：我需要一个 `CurrentUser`。如果依赖函数抛出 `HTTPException`，路由函数不会继续执行，FastAPI 会直接返回错误响应。

## 五、依赖可以继续依赖别的依赖

FastAPI 会解析一棵依赖树。比如“管理员接口”可以先拿当前用户，再检查用户角色。路由函数只接收最终需要的结果。

In [ ]:
def require_admin(user: UserDep) -> CurrentUser:
    if user.role != "admin":
        raise HTTPException(status_code=403, detail="需要管理员权限")
    return user


AdminUserDep = Annotated[CurrentUser, Depends(require_admin)]


@app.get("/admin/reports")
def list_admin_reports(user: AdminUserDep, pagination: Pagination):
    return {
        "viewer": user.username,
        "reports": ["daily-risk", "weekly-growth"],
        **pagination,
    }


client = TestClient(app)
client.get(
    "/admin/reports?limit=1",
    headers={"Authorization": "Bearer dev-token"},
).json()

可以把这个过程想成一条请求流水线：

```mermaid
flowchart LR
    A[HTTP 请求] --> B[get_current_user]
    B --> C[require_admin]
    A --> D[get_pagination]
    C --> E[list_admin_reports]
    D --> E
    E --> F[HTTP 响应]
```

接口函数不用关心这些步骤如何排序。它声明了依赖，FastAPI 负责解析依赖图。

## 六、测试时可以覆盖依赖

`Depends` 的另一个实用点是测试。真实认证逻辑可能依赖外部服务或数据库，单元测试不一定想碰它。FastAPI 提供 `app.dependency_overrides`，可以把某个依赖临时替换成测试版本。

In [ ]:
def fake_current_user() -> CurrentUser:
    return CurrentUser(username="test-user", role="admin")


app.dependency_overrides[get_current_user] = fake_current_user

client = TestClient(app)
response = client.get("/admin/reports?limit=2")
print(response.status_code)
print(response.json())

app.dependency_overrides.clear()

这个替换只针对依赖函数本身。路由函数不用改，测试仍然走真实的 FastAPI 请求解析、依赖解析和响应生成。

## 七、一个常见误区：Depends 里传函数，不是传函数调用结果

正确写法是：

```python
user: Annotated[CurrentUser, Depends(get_current_user)]
```

不要写成：

```python
user: Annotated[CurrentUser, Depends(get_current_user())]
```

前者是把“如何获取当前用户”这件事交给 FastAPI；后者是在 Python 导入模块时就调用函数，既拿不到请求对象里的 header，也会破坏 FastAPI 的依赖解析。

## 八、什么时候该抽成依赖

可以用一个简单判断：如果某段逻辑是“接口运行前的准备或检查”，并且未来可能被多个接口复用，就适合写成依赖。

常见场景包括：

- 统一分页、排序、过滤参数。
- 获取当前用户、检查权限。
- 创建和关闭数据库会话。
- 读取配置、租户、请求上下文。
- 在测试中替换外部服务。

先别急着把所有函数都改成依赖。普通业务计算函数继续保持普通函数就好。`Depends` 最擅长处理的是“请求进入接口前，FastAPI 能帮你准备好的东西”。

## 小结

`Depends` 可以先记成三句话：

- 依赖函数声明自己需要从请求里拿什么，并返回路由函数需要的对象。
- 路由函数用 `Annotated[类型, Depends(依赖函数)]` 声明自己需要这个对象。
- 每次请求进来时，FastAPI 负责调用依赖、处理校验和异常、把返回值注入路由函数。

理解到这里，看到 `from fastapi import Depends` 就不陌生了。它不是让代码变玄学，而是把重复的准备工作放到一个明确、可复用、可测试的位置。